In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [24]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import InMemoryVectorStore
from langchain.tools import tool
from langchain.agents import create_agent

In [3]:
loader=PyPDFLoader("../data/medical_report.pdf")
docs=loader.load()

In [4]:
splitter=RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splitted_docs=splitter.split_documents(docs)

In [7]:
embeddings=OpenAIEmbeddings(model="text-embedding-3-small")
vector_store=InMemoryVectorStore.from_documents(
    documents=splitted_docs,
    embedding=embeddings
)

###agent creation (tools,llm, promt ) 

In [21]:
@tool

def retriever_tool(query:str):
    """
        This tool can help you to retrieve the relevant data of the PDF Documents, and these pdf
        documents have details about medical reports.
    """
    docs=vector_store.similarity_search(query=query, k=4)
    context=""

    for doc in docs:
        context =doc.page_content + "\n\n"

    return context

In [22]:
llm=ChatOpenAI(model="gpt-5")

In [23]:
System_Prompt = """
    You are a helpful assistant that answers questions using retrieved context.
	ALWAYS use the `retriever_tool` tool for questions requiring external knowledge.
"""

In [25]:
agent=create_agent(
    model=llm,
    tools=[retriever_tool],
    system_prompt=System_Prompt
)

In [29]:
query="what is the name of patient and what is te name of doctors"
res=agent.invoke({"messages":[{"role": "user", "content":query}]})

In [27]:
result=res["messages"][-1].content

In [30]:
print(result)

- Patient name: Ms. Nikita Chudhary
- Doctor name (Referring): Dr. Nitin Nahar

Note: The report also lists lab doctors (e.g., Dr. Swapnil Gupta, Dr. Kiran Bhargava Pathak). If you meant a different doctor type, tell me which one.
